# Regional European Daily SST (OSTIA) Analysis: Identifying Marine Extremes with `MarEx-Detect`

This notebook turns 40 years (1982-2022) of daily OSTIA sea-surface temperature at its full 0.05° resolution over Europe (30°N-70°N, 20°W-45°E: the Atlantic around the UK and Ireland, the North Sea and Baltic, the Mediterranean and part of the Black Sea) into a boolean record of marine heatwaves with two combinations of anomaly and threshold method. The box is 800 × 1300 cells, about the size of the whole global 0.25° grid, so it needs about the same resources as the global example:

| | Anomaly | Threshold |
|---|---|---|
| Part 1 | `shifting_baseline`: anomaly from a smoothed day-of-year climatology of the preceding `window_years` (15) | `seasonal_percentile`: day-of-year 95th percentile, after Hobday et al. (2016) |
| Part 2 | `detrend_harmonic`: removes the mean, the annual & semi-annual harmonics, and a linear + quadratic trend | `global_percentile`: one 95th percentile per cell over all days |

Part 3 compares the two. The shifting baseline consumes its first `window_years` as spin-up, so Part 1's record starts
15 years after the input does. Part 1's output feeds the tracking example, `02_id_track_events.ipynb`.

Everything runs on a single node with 64 cores and 128 GB. Each stage prints its wall time and peak memory as it
finishes, and the table at the end collects them. The other two anomaly methods (`fixed_baseline` and
`detrend_fixed_baseline`) and the trade-offs between all four are described under
[Choosing an Anomaly Method](https://marex.readthedocs.io/en/latest/guide/anomalies.html#choosing-a-method).

In [ ]:
from getpass import getuser
from pathlib import Path

import dask
import intake
import numpy as np
import xarray as xr

import marEx
import marEx.helper as hpc

In [ ]:
scratch_dir = Path("/scratch") / getuser()[0] / getuser()  # Lustre scratch
out_dir = scratch_dir / "marEx" / "examples" / "regional"

lat_range, lon_range = slice(30, 70), slice(-20, 45)  # 30°N-70°N, 20°W-45°E
period = slice("1982-01-01", "2022-05-31")  # 40 years of input
window_years = 15  # shifting-baseline length in years. Part 1's output starts this many years after `period`
plot_window = slice("2016-01-01", "2021-12-31")  # x-range of the time-series figures

# One node: 4 workers x 16 threads x 28 GB = 64 cores, 112 GB for the workers, the rest of 128 GB for the client
n_workers, threads_per_worker, memory_limit = 4, 16, "28GB"

In [ ]:
out_dir.mkdir(parents=True, exist_ok=True)

# memory_limit is passed explicitly because on an HPC node Dask sizes workers from the whole node's memory,
# not from the job's allocation, and without it the workers can claim more than the job was granted.
client = hpc.start_local_cluster(
    n_workers=n_workers,
    threads_per_worker=threads_per_worker,
    memory_limit=memory_limit,
    scratch_dir=scratch_dir / "marEx" / "dask",
)

# Records wall time, peak memory, spill and worker restarts for each `monitor.stage(...)` block below
monitor = hpc.ResourceMonitor(client)

## Pre-Process with Shifting-Baseline & Hobday et al. (2016) Definition for Extremes

### A note on chunking

Both methods read the input with one chunking: a small time chunk and the whole domain in each chunk
(`{"time": 30, "lat": -1, "lon": -1}`). The climatology and percentile reductions are global in time, so a task's
memory is `n_time × (cells in one spatial chunk) × 4 B`, and marEx tiles the spatial dimension internally wherever a
reduction needs it. A large time chunk would only inflate each input chunk to several GB. The sizing rules are under
[Memory Management and Chunking](https://marex.readthedocs.io/en/latest/guide/performance.html#chunking).

In [ ]:
# Daily OSTIA at full resolution, from the EERIE intake catalogue on Levante
cat = intake.open_catalog("https://raw.githubusercontent.com/eerie-project/intake_catalogues/main/eerie.yaml")
ostia = cat["dkrz.disk.observations.OSTIA.OSTIA.daily"]

sst = (
    ostia(chunks={})
    .to_dask()
    .sst.astype(np.float32)
    .sel(lat=lat_range, lon=lon_range, time=period)
    .chunk({"time": 30, "lat": -1, "lon": -1})
)
sst

In [ ]:
with monitor.stage("Detect: shifting baseline + Hobday"):
    extremes_ds = marEx.preprocess_data(
        sst,
        method_anomaly="shifting_baseline",  # anomaly from a backward-looking climatology of the previous `window_years`
        method_extreme="seasonal_percentile",  # day-of-year threshold (Hobday et al., 2016)
        threshold_percentile=95,
        window_years=window_years,
        smooth_days=21,  # smoothing applied to the climatology (days)
        window_days=11,  # samples from an 11-day window around each day of year enter its percentile
        dimensions={"time": "time", "x": "lon", "y": "lat"},  # a "y" dimension tells marEx this is a lat/lon grid
        dask_chunks={"time": 25},  # chunking of the output
    )
    extremes_ds.to_zarr(out_dir / "extremes_shifting_hobday.zarr", mode="w")
extremes_ds

## Pre-Process with Detrend Harmonic Baseline & Global-in-Time Threshold Value

The same input, so the `sst` loaded above is reused.

In [ ]:
with monitor.stage("Detect: harmonic detrend + global threshold"):
    extremes_ds = marEx.preprocess_data(
        sst,
        method_anomaly="detrend_harmonic",  # removes the mean, annual & semi-annual harmonics and the trend
        method_extreme="global_percentile",  # one threshold per cell, over all days
        threshold_percentile=95,
        detrend_orders=[1, 2],  # linear + quadratic trend
        dimensions={"time": "time", "x": "lon", "y": "lat"},
        dask_chunks={"time": 25},
    )
    extremes_ds.to_zarr(out_dir / "extremes_harmonic_global.zarr", mode="w")
extremes_ds

# Compare Results of Pre-Processing Methods

In [ ]:
import calendar

import cartopy.crs as ccrs
import dask.array as da
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

In [ ]:
extremes_ds_shifting = xr.open_zarr(out_dir / "extremes_shifting_hobday.zarr", chunks={})

# Compare over the shifting-baseline record, which starts `window_years` after the input
time_slice = slice(extremes_ds_shifting.time.values[0], extremes_ds_shifting.time.values[-1])
sst = (
    ostia(chunks={})
    .to_dask()
    .sst.astype(np.float32)
    .sel(lat=lat_range, lon=lon_range, time=time_slice)
    .chunk({"time": 75, "lat": "auto", "lon": "auto"})
)

extremes_ds_shifting = extremes_ds_shifting.sel(time=time_slice).chunk({"time": 75, "lat": "auto", "lon": "auto"})
extremes_ds_detrend = (
    xr.open_zarr(out_dir / "extremes_harmonic_global.zarr", chunks={})
    .sel(time=time_slice)
    .chunk({"time": 75, "lat": "auto", "lon": "auto"})
)

## Anomaly Distributions and Variance

The positive shift of the shifting-baseline anomalies comes from the backward-looking climatology. It is intended: the
baseline is the recent past that an adapting organism has experienced.

The harmonic fit (the mean plus annual & semi-annual harmonics) cannot represent the shape of the seasonal cycle
everywhere. Combined with one threshold per cell (`global_percentile`), this produces unphysical seasonal biases in the
variance, although the total variability of the two methods is similar.

In [ ]:
bins = np.arange(-4, 4, 0.01)
# Focus on North Atlantic subregion
lat_slice = slice(45, 55)
lon_slice = slice(-20, 0)

dat_shifting = extremes_ds_shifting.dat_anomaly.sel(lat=lat_slice, lon=lon_slice).data
dat_anomaly = extremes_ds_detrend.dat_anomaly.sel(lat=lat_slice, lon=lon_slice).data

hist_shifting, _ = da.histogram(dat_shifting, bins=bins, density=True)
hist_detrend, _ = da.histogram(dat_anomaly, bins=bins, density=True)

hist_shifting_computed, hist_detrend_computed = dask.compute(hist_shifting, hist_detrend)

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True, gridspec_kw={"hspace": 0.25})

axes[0].plot(bins[:-1], hist_shifting_computed, label="Shifting+Hobday", alpha=0.7)
axes[0].plot(bins[:-1], hist_detrend_computed, label="Harmonic+Global", alpha=0.7)
axes[0].set_ylabel("Density")
axes[0].set_title("Histogram of Anomaly (North Atlantic Subregion)")
axes[0].set_xlim(-4, 4)
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].axvline(0, color="black", linestyle="--", linewidth=0.8)
axes[0].legend()

axes[1].plot(bins[:-1], hist_shifting_computed, label="Shifting+Hobday", alpha=0.7)
axes[1].plot(bins[:-1], hist_detrend_computed, label="Harmonic+Global", alpha=0.7)
axes[1].set_xlabel("SST Anomaly (K)")
axes[1].set_ylabel("Density")
axes[1].set_xlim(-4, 4)
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].set_yscale("log")
axes[1].axvline(0, color="black", linestyle="--", linewidth=0.8)

plt.tight_layout()
plt.show()

In [ ]:
shifting_variance = extremes_ds_shifting.dat_anomaly.std(dim="time")
detrend_variance = extremes_ds_detrend.dat_anomaly.std(dim="time")

config = marEx.PlotConfig(var_units="Anomaly STD (K)", cmap="hot_r", cperc=[0, 96], grid_labels=True, projection=ccrs.PlateCarree())

combined_variance = xr.concat([shifting_variance, detrend_variance], dim="method")
combined_variance = combined_variance.assign_coords(method=["Shifting+Hobday", "Harmonic+Global"])

fig, axes = combined_variance.plotX.multi_plot(config, col="method", col_wrap=2)

In [ ]:
shifting_seasonal_std = extremes_ds_shifting.extreme_events.groupby("time.season").std(dim="time").persist()
detrend_seasonal_std = extremes_ds_detrend.extreme_events.groupby("time.season").std(dim="time").persist()

# Add method coordinate and concatenate
shifting_seasonal_std = shifting_seasonal_std.expand_dims({"method": ["Shifting+Hobday"]})
detrend_seasonal_std = detrend_seasonal_std.expand_dims({"method": ["Harmonic+Global"]})
combined_seasonal_std = xr.concat([shifting_seasonal_std, detrend_seasonal_std], dim="method")

config = marEx.PlotConfig(
    var_units="Anomaly STD (K)", cmap="hot_r", clim=[0, 0.4], grid_labels=False, show_colorbar=True, projection=ccrs.PlateCarree()
)

seasons = combined_seasonal_std["season"].values

for season in seasons:
    data = combined_seasonal_std.sel(season=season)
    fig, axes = data.plotX.multi_plot(config, col="method", col_wrap=2)
    fig.suptitle(f"Season: {season}", fontsize=14)
    plt.show()

## Regional Extreme Event Frequency

Under the Hobday definition, extremes are by construction relative to the time of year. The seasonal pattern in the
harmonic-detrended frequency therefore comes from the non-harmonic shape of the seasonal cycle (cf. the time series
above).

In [ ]:
shifting_frequency = extremes_ds_shifting.extreme_events.mean(dim="time")
detrend_frequency = extremes_ds_detrend.extreme_events.mean(dim="time")

config = marEx.PlotConfig(var_units="MHW Frequency", cmap="hot_r", cperc=[0, 96], grid_labels=True, projection=ccrs.PlateCarree())

combined_frequency = xr.concat([shifting_frequency, detrend_frequency], dim="method")
combined_frequency = combined_frequency.assign_coords(method=["Shifting+Hobday", "Harmonic+Global"])

fig, axes = combined_frequency.plotX.multi_plot(config, col="method", col_wrap=2)

In [ ]:
shifting_seasonal_frequency = extremes_ds_shifting.extreme_events.groupby("time.season").mean(dim="time").persist()
detrend_seasonal_frequency = extremes_ds_detrend.extreme_events.groupby("time.season").mean(dim="time").persist()

# Add method coordinate and concatenate
shifting_seasonal_frequency = shifting_seasonal_frequency.expand_dims({"method": ["Shifting+Hobday"]})
detrend_seasonal_frequency = detrend_seasonal_frequency.expand_dims({"method": ["Harmonic+Global"]})
combined_seasonal_frequency = xr.concat([shifting_seasonal_frequency, detrend_seasonal_frequency], dim="method")

config = marEx.PlotConfig(
    var_units="MHW Frequency", cmap="hot_r", clim=[0, 0.08], grid_labels=False, show_colorbar=True, projection=ccrs.PlateCarree()
)

seasons = combined_seasonal_frequency["season"].values

for season in seasons:
    data = combined_seasonal_frequency.sel(season=season)
    fig, axes = data.plotX.multi_plot(config, col="method", col_wrap=2)
    fig.suptitle(f"Season: {season}", fontsize=14)
    plt.show()

In [ ]:
# Regional analysis: North Atlantic vs Mediterranean
monthly_shifting_north = extremes_ds_shifting.extreme_events.sel(lat=slice(50, 70)).mean(dim=["lat", "lon"]).rolling(time=60).mean()
monthly_detrend_north = extremes_ds_detrend.extreme_events.sel(lat=slice(50, 70)).mean(dim=["lat", "lon"]).rolling(time=60).mean()

# Mediterranean region
monthly_shifting_med = (
    extremes_ds_shifting.extreme_events.sel(lat=slice(30, 49))
    .sel(lon=slice(0, 45))
    .mean(dim=["lat", "lon"])
    .rolling(time=60)
    .mean()
)
monthly_detrend_med = (
    extremes_ds_detrend.extreme_events.sel(lat=slice(30, 49)).sel(lon=slice(0, 45)).mean(dim=["lat", "lon"]).rolling(time=60).mean()
)

plt.figure(figsize=(12, 6))
plt.plot(
    monthly_shifting_north.time,
    monthly_shifting_north,
    label="Shifting+Hobday (North)",
)
plt.plot(
    monthly_detrend_north.time,
    monthly_detrend_north,
    label="Harmonic+Global (North)",
    alpha=0.5,
)
plt.plot(
    monthly_shifting_med.time,
    monthly_shifting_med,
    label="Shifting+Hobday (Mediterranean)",
)
plt.plot(
    monthly_detrend_med.time,
    monthly_detrend_med,
    label="Harmonic+Global (Mediterranean)",
    alpha=0.5,
)
plt.grid(True, which="major", linestyle="--", alpha=0.5)
plt.grid(True, which="minor", linestyle=":", alpha=0.4, axis="x")
plt.xlabel("Time")
plt.ylabel("Frequency of Extreme Events")
plt.xlim(np.datetime64(plot_window.start), np.datetime64(plot_window.stop))
plt.ylim(-0.01, 0.15)
plt.gca().xaxis.set_minor_locator(mdates.MonthLocator(bymonth=range(2, 13)))
plt.legend()
plt.show()

In [ ]:
# Daily climatology split by European regions
monthly_shifting_north = (
    extremes_ds_shifting.extreme_events.sel(lat=slice(50, 70))
    .mean(dim=["lat", "lon"])
    .rolling(time=10, center=True)
    .mean("time")
    .groupby("time.dayofyear")
    .mean(dim="time")
    .isel(dayofyear=slice(0, -2))
)
monthly_shifting_med = (
    extremes_ds_shifting.extreme_events.sel(lat=slice(30, 49))
    .sel(lon=slice(0, 45))
    .mean(dim=["lat", "lon"])
    .rolling(time=10, center=True)
    .mean("time")
    .groupby("time.dayofyear")
    .mean(dim="time")
    .isel(dayofyear=slice(0, -2))
)
monthly_detrend_north = (
    extremes_ds_detrend.extreme_events.sel(lat=slice(50, 70))
    .mean(dim=["lat", "lon"])
    .rolling(time=10, center=True)
    .mean("time")
    .groupby("time.dayofyear")
    .mean(dim="time")
    .isel(dayofyear=slice(0, -2))
)
monthly_detrend_med = (
    extremes_ds_detrend.extreme_events.sel(lat=slice(30, 49))
    .sel(lon=slice(0, 45))
    .mean(dim=["lat", "lon"])
    .rolling(time=10, center=True)
    .mean("time")
    .groupby("time.dayofyear")
    .mean(dim="time")
    .isel(dayofyear=slice(0, -2))
)

days = monthly_shifting_north.dayofyear.values
month_starts = [1] + list(np.cumsum([calendar.monthrange(2001, m)[1] for m in range(1, 12)]))
month_labels = [calendar.month_abbr[m] for m in range(1, 13)]

plt.figure(figsize=(10, 5))
plt.plot(days, monthly_shifting_north, label="Shifting+Hobday (North)")
plt.plot(days, monthly_detrend_north, label="Harmonic+Global (North)", alpha=0.5)
plt.plot(days, monthly_shifting_med, label="Shifting+Hobday (Mediterranean)")
plt.plot(days, monthly_detrend_med, label="Harmonic+Global (Mediterranean)", alpha=0.5)
plt.grid(True, linestyle="--", alpha=0.5)
plt.xlabel("Month")
plt.ylabel("Frequency of Extreme Events")
plt.title("Daily Climatology of Extreme Events (European Regional Split)")
plt.legend()

plt.xticks(month_starts, month_labels)
plt.xlim(1, 366)
plt.tight_layout()
plt.show()

## Wall Time and Memory

One row per stage, measured while the notebook ran on the cluster configured at the top. Peak worker memory is the
summed process memory the workers report to the scheduler, which is the quantity Dask compares against `memory_limit`.
The client (this kernel) is counted separately.

In [ ]:
monitor.summary()

In [ ]:
client.close()